# 3단계: Q2 — 정상 학습 이미지를 줄이면 떨어지는 폭이 데이터셋마다 다른가

Q1 1위 설정을 데이터셋마다 모은 넷으로 학습 장수를 50장, 10장으로 줄여 돌린다. 전처리는 Q1과 같은 비교 단계(`Resize((256, 256))`, 자르지 않음)다.

| 방법 | 설정 | Q1에서 1위였던 곳 |
|---|---|---|
| PatchCore | WRN-50-2 (2,3) 10% | MVTec AD |
| PatchCore | WRN-50-2 (1,2,3) 1% | Real-IAD, VisA |
| PaDiM | WRN-50-2 (2,3), 차원 550 | MVTec AD, Real-IAD |
| PaDiM | WRN-50-2 (1,2,3), 차원 550 | VisA (차이 0.19점, 시드 표준편차 0.52·0.61) |

**돌리는 칸.** 설정 4 x 37 부류(MVTec 15, VisA 12, Real-IAD 10) x 시드 3 x 학습 장수 2(50, 10) = 888번. 전체 장수 칸은 Q1 결과(레포 `results/q1_*.csv`)를 그대로 쓴다.

**표본 고르기.** 시드마다 학습 목록에서 중복 없이 뽑는다(`data.subsample_train`). 같은 시드의 10장은 50장 안에 든다. 시드가 바꾸는 것은 고른 사진과 코어셋 시작점(PaDiM은 무작위 차원) 둘이라, 시드 표준편차에 둘이 섞여 있다. MVTec AD Toothbrush는 학습이 60장이라 50장이 전체의 83%다.

**PaDiM 공분산.** 위치마다 공분산을 사진 N장으로 추정하므로 계수가 N−1(50장이면 49, 10장이면 9)을 넘지 못하고 나머지 550−(N−1) 방향은 ε = 0.01만으로 채워진다(README 「Q1 설정」).

**시간.** Q1 기록으로 어림하면 6.8~8.6시간이다. 아래 끝은 예측·지표 시간만 더한 값(시험 사진 수는 그대로다), 위 끝은 학습 시간을 장수에 비례해 줄인 값이다. 코어셋 시간은 뱅크 크기의 제곱으로 줄어서 실제 학습 시간은 위 끝 계산보다 짧을 것으로 본다. 잰 값이 아니다.

**순서.** 시드 하나마다 MVTec -> Real-IAD -> VisA, 데이터셋마다 50장 -> 10장으로 돈다.

**끊겨도 된다.** 결과는 부류 하나가 끝날 때마다 Drive `beyond_mvtec_results/q2_*.csv`에 적는다. 런타임이 끊기면 1·2번 셀과 4번의 첫 셀을 다시 돌린 뒤 5번 셀을 다시 돌리면 끝난 줄을 건너뛴다.

**실행 전 준비.** 런타임 GPU(L4), 보안 비밀 `GITHUB_TOKEN` 접근 허용. `configs/experiments.yaml`의 `few_shot.settings`가 GitHub main에 올라가 있어야 1번 셀을 넘어간다.

## 1. 코드와 패키지

In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/beyond-mvtec"
REPO_URL = "https://github.com/chlee1027/beyond-mvtec.git"
UPSTREAM_DIR = "/content/patchcore-inspection"
UPSTREAM_COMMIT = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"

from google.colab import userdata
token = userdata.get("GITHUB_TOKEN").strip()
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "-q", auth_url, REPO_DIR], capture_output=True, text=True)
else:
    # 이미 받아 둔 레포는 GitHub 의 최신 main 으로 맞춘다(토큰 든 주소는 저장하지 않는다)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only", auth_url, "main"],
                       capture_output=True, text=True)
# 실패해도 오류 메시지에 토큰이 찍히지 않게 가린다
assert r.returncode == 0, "레포 받기 실패: " + r.stderr.replace(token, "***")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
if not os.path.isdir(UPSTREAM_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/amazon-science/patchcore-inspection.git", UPSTREAM_DIR], check=True)
subprocess.run(["git", "-C", UPSTREAM_DIR, "checkout", "-q", UPSTREAM_COMMIT], check=True)

!pip uninstall -y -q sympy
!pip install -q sympy==1.13.3 faiss-gpu-cu12 timm click pretrainedmodels scikit-image scikit-learn scipy tqdm pandas pyyaml

sys.path[:0] = [os.path.join(REPO_DIR, "src"), os.path.join(UPSTREAM_DIR, "src")]
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import faiss, torch, torchvision
assert torch.cuda.is_available(), "GPU 런타임이 아니다"
print(f"레포 {COMMIT} | Python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | faiss {faiss.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert hasattr(faiss, "StandardGpuResources"), "faiss 의 GPU 판이 아니다"

from beyond_mvtec import data, experiment as ex, patchcore_adapter
config = ex.load_config(f"{REPO_DIR}/configs/experiments.yaml")
Q2 = config.get("few_shot", {})
# 레포가 Q2 설정을 적기 전 판이면 여기서 멈춘다
assert "settings" in Q2 and hasattr(patchcore_adapter, "_releasing_sampler") \
    and config["datasets"]["real_iad"].get("split") == "sv", \
    "옛 코드가 읽혔다. Q2 설정을 push 했는지 확인하고, 런타임 > 세션 다시 시작 후 1번 셀부터 다시 돌린다"
SETTINGS = [s for s in ex.comparison_settings(config) if s["setting"] in Q2["settings"]]
assert sorted(s["setting"] for s in SETTINGS) == sorted(Q2["settings"]) and len(SETTINGS) == 4, Q2["settings"]
N_TRAIN = [int(n) for n in Q2["n_train"] if str(n) != "all"]
assert N_TRAIN == [50, 10], N_TRAIN
print("설정", len(SETTINGS), "개:", [s["setting"] for s in SETTINGS], "| 학습 장수", N_TRAIN, "(전체는 Q1 결과)")

## 2. Drive와 데이터

In [ ]:
import shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")

MY = "/content/drive/MyDrive"
os.environ["TORCH_HOME"] = f"{MY}/torch_cache"
RESULT_DIR = f"{MY}/beyond_mvtec_results"
os.makedirs(RESULT_DIR, exist_ok=True)

def copy_once(src, dst):
    """Drive 에서 파일을 하나씩 읽으면 느려서 런타임 디스크로 한 번 옮긴다. 진행 표시는 없다."""
    if not os.path.isfile(f"{dst}/.copied"):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/.copied", "w").close()
    print("Drive:", src, "-> 로컬:", dst)

MVTEC_DRIVE = next((p for p in [f"{MY}/anormaly_detection/mvtec", f"{MY}/mvtec", f"{MY}/MVTecAD"]
                    if os.path.isdir(f"{p}/bottle")), None)
VISA_DRIVE = next((p for p in [f"{MY}/visa", f"{MY}/VisA_20220922", f"{MY}/VisA"]
                   if os.path.isfile(f"{p}/split_csv/1cls.csv")), None)
assert MVTEC_DRIVE and VISA_DRIVE, (MVTEC_DRIVE, VISA_DRIVE)

ROOTS = {"mvtec_ad": "/content/data/mvtec_ad", "visa": "/content/data/visa", "real_iad": "/content/data/real_iad"}
copy_once(MVTEC_DRIVE, ROOTS["mvtec_ad"])
copy_once(VISA_DRIVE, ROOTS["visa"])

# Real-IAD: 분할 JSON 을 풀고, 부류 zip 에서 C1 사진과 마스크만 푼다
RI = config["datasets"]["real_iad"]
VIEW, RI_SPLIT = RI["views"][0], RI["split"]
os.makedirs(ROOTS["real_iad"], exist_ok=True)
if not os.path.isdir(f"{ROOTS['real_iad']}/realiad_jsons"):
    with zipfile.ZipFile(f"{MY}/realiad_jsons.zip") as z:
        z.extractall(f"{ROOTS['real_iad']}/realiad_jsons")
for category in RI["categories"]:
    data.prepare_real_iad(f"{MY}/Real_IAD/{category}.zip", ROOTS["real_iad"], category, view=VIEW)
print("Real-IAD", len(RI["categories"]), "부류:", RI["categories"], "| 시점", VIEW, "| 분할", RI_SPLIT)

CATEGORIES = {"mvtec_ad": data.MVTEC_CATEGORIES, "visa": data.visa_categories(ROOTS["visa"]),
              "real_iad": RI["categories"]}

In [ ]:
import pandas as pd

rows = []
for dataset, categories in CATEGORIES.items():
    for category in categories:
        split = ex.list_split(dataset, ROOTS[dataset], category, VIEW, real_iad_split=RI_SPLIT)
        rows.append({"dataset": dataset, "category": category, "train": len(split.train),
                     "test": len(split.test), "test_anomaly": sum(s.label for s in split.test)})
counts = pd.DataFrame(rows)
display(counts.groupby("dataset")[["train", "test", "test_anomaly"]].sum())

# 00 노트북에서 센 장수와 같아야 한다
total = counts.groupby("dataset")[["train", "test"]].sum()
assert tuple(total.loc["mvtec_ad"]) == (3629, 1725), tuple(total.loc["mvtec_ad"])
assert tuple(total.loc["visa"]) == (8659, 2162), tuple(total.loc["visa"])
expected_ri = {"bottle_cap": (370, 633), "pcb": (279, 723), "phone_battery": (350, 650), "porcelain_doll": (402, 598),
               "regulator": (477, 543), "sim_card_set": (305, 701), "switch": (267, 743), "usb": (354, 656),
               "wooden_beads": (305, 696), "woodstick": (443, 558)}
got_ri = {r.category: (r.train, r.test) for r in counts[counts.dataset == "real_iad"].itertuples()}
assert got_ri == expected_ri, got_ri
print("장수가 00 노트북과 같다")
display(counts[counts.dataset == "real_iad"])

## 3. 전체 장수 칸(Q1 결과) 확인

레포 `results/`의 Q1 CSV에서 이 노트북의 설정 넷만 꺼낸다. 칸이 모자라면 멈춘다.

In [ ]:
Q1_ROWS = pd.concat([pd.read_csv(f"{REPO_DIR}/results/q1_{d}.csv") for d in ROOTS], ignore_index=True)
Q1_ROWS = Q1_ROWS[Q1_ROWS.setting.isin(Q2["settings"]) & (Q1_ROWS.n_train_requested.astype(str) == "all")]
for dataset, categories in CATEGORIES.items():
    part = Q1_ROWS[Q1_ROWS.dataset == dataset]
    assert len(part) == len(SETTINGS) * len(categories) * len(config["seeds"]), (dataset, len(part))
    assert set(part.category) == set(categories), dataset
print("Q1 전체 장수 칸", len(Q1_ROWS), "줄 | 커밋", sorted(Q1_ROWS.commit.astype(str).unique()))

## 4. 짧은 실행: 표본 확인과 10장 한 부류

먼저 37 부류 x 시드 3에서 10장이 50장 안에 드는지, 고른 장수가 맞는지 센다(학습 없음). 그다음 VisA `pcb3` 시드 0의 10장으로 설정 넷을 돌린다. 결과는 5번과 같은 CSV에 적히므로 5번에서 다시 돌지 않는다.

**넘어가는 기준.** 네 줄 모두 오류 없이 끝나고 `n_train`이 10이면 5번으로 간다. 오류가 나면 5번을 돌리지 말고 이 셀 출력을 보낸다.

In [ ]:
PREPROCESS = config["preprocess"]["compare"]
OPTIONS = dict(device="cuda", batch_size=8, num_workers=4, faiss_on_gpu=True, commit=COMMIT,
               view=VIEW, real_iad_split=RI_SPLIT, save_dir=f"{RESULT_DIR}/q2_scores")
CSV = {d: f"{RESULT_DIR}/q2_{d}.csv" for d in ROOTS}

In [ ]:
for dataset, categories in CATEGORIES.items():
    for category in categories:
        train = ex.list_split(dataset, ROOTS[dataset], category, VIEW, real_iad_split=RI_SPLIT).train
        for seed in config["seeds"]:
            picked = {n: data.subsample_train(train, n, seed) for n in N_TRAIN}
            assert all(len(picked[n]) == n for n in N_TRAIN), (dataset, category, seed)
            assert set(picked[10]) <= set(picked[50]), (dataset, category, seed)
print("37 부류 x 시드 3: 장수가 맞고 10장이 50장 안에 든다")

ex.run_grid(SETTINGS, "visa", ROOTS["visa"], ["pcb3"], [0], "compare", PREPROCESS, CSV["visa"],
            n_train=10, **OPTIONS)
done = pd.read_csv(CSV["visa"])
display(done[(done.category == "pcb3") & (done.seed == 0) & (done.n_train_requested.astype(str) == "10")]
        [["setting", "n_train", "image_auroc", "pixel_auroc", "pro", "bank_size", "fit_sec", "predict_sec",
          "metric_sec", "peak_gpu_gb"]])

## 5. Q2 실행

점수와 라벨은 `q2_scores/`에 남기고 이상 지도는 남기지 않는다.

In [ ]:
for seed in config["seeds"]:
    for dataset in ["mvtec_ad", "real_iad", "visa"]:
        for n_train in N_TRAIN:
            print(f"===== 시드 {seed} | {dataset} | 학습 {n_train}장 =====")
            ex.run_grid(SETTINGS, dataset, ROOTS[dataset], CATEGORIES[dataset], [seed], "compare", PREPROCESS,
                        CSV[dataset], n_train=n_train, **OPTIONS)

## 6. 결과

Q1 전체 장수 칸과 합쳐 시드마다 부류 평균을 내고, 시드 셋의 평균 ± 표준편차(n−1)를 적는다. 단위는 %다.

- **값 표**: 설정 x (데이터셋, 학습 장수).
- **떨어진 폭**: 시드마다 (전체 − N장)을 낸 뒤 평균 ± 표준편차. 같은 시드끼리 뺀다.
- **순위**: (데이터셋, 학습 장수)마다 설정 넷의 순위(1 = 가장 높음). Q1 1위 설정이 장수를 줄여도 1위인지 본다.

시드 0만 끝난 상태에서도 돌릴 수 있다(표준편차는 비어 나온다).

In [ ]:
q2_paths = [p for p in CSV.values() if os.path.isfile(p)]
rows = pd.concat([Q1_ROWS] + [pd.read_csv(p) for p in q2_paths], ignore_index=True)
rows["n_train_requested"] = rows.n_train_requested.astype(str)
ORDER = ["all"] + [str(n) for n in N_TRAIN]
METRICS = ["image_auroc", "pixel_auroc", "pro"]

per_seed = rows.groupby(["dataset", "setting", "n_train_requested", "seed"]).agg(
    **{m: (m, "mean") for m in METRICS}, n_categories=("category", "nunique")).reset_index()
per_seed[METRICS] *= 100
expected_n = {d: len(c) for d, c in CATEGORIES.items()}
cells = per_seed.groupby(["dataset", "setting", "n_train_requested"]).agg(
    n_seeds=("seed", "nunique"), n_categories=("n_categories", "min")).reset_index()
complete = len(cells) == len(SETTINGS) * len(ORDER) * len(CATEGORIES) and all(
    (r.n_seeds == 3) and (r.n_categories == expected_n[r.dataset]) for r in cells.itertuples())
print("칸이 다 찼나:", complete)

def mean_std(frame, value):
    g = frame.groupby(["setting", "dataset", "n_train_requested"])[value]
    out = g.mean().round(2).astype(str) + " ± " + g.std(ddof=1).round(2).astype(str)
    return out.unstack(["dataset", "n_train_requested"]).reindex(
        columns=pd.MultiIndex.from_product([list(CATEGORIES), ORDER]))

for m in METRICS:
    print(m, "(%, 시드 평균 ± 표준편차)")
    display(mean_std(per_seed, m))

In [ ]:
# 떨어진 폭: 같은 시드의 전체 장수 값에서 뺀다
full = per_seed[per_seed.n_train_requested == "all"].set_index(["dataset", "setting", "seed"])[METRICS]
drop = per_seed[per_seed.n_train_requested != "all"].join(full, on=["dataset", "setting", "seed"], rsuffix="_all")
for m in METRICS:
    drop[f"{m}_drop"] = drop[f"{m}_all"] - drop[m]
for m in ["image_auroc"]:
    print("이미지 AUROC 떨어진 폭 (전체 − N장, %p, 시드 평균 ± 표준편차)")
    g = drop.groupby(["setting", "dataset", "n_train_requested"])[f"{m}_drop"]
    display((g.mean().round(2).astype(str) + " ± " + g.std(ddof=1).round(2).astype(str))
            .unstack(["dataset", "n_train_requested"])
            .reindex(columns=pd.MultiIndex.from_product([list(CATEGORIES), ORDER[1:]])))

# 순위: (데이터셋, 장수)마다 설정 넷
values = per_seed.groupby(["setting", "dataset", "n_train_requested"]).image_auroc.mean().unstack(
    ["dataset", "n_train_requested"]).reindex(columns=pd.MultiIndex.from_product([list(CATEGORIES), ORDER]))
print("이미지 AUROC 순위 (1 = 가장 높음, 설정 넷 안에서)")
display(values.rank(ascending=False, method="average"))

In [ ]:
# 시간과 메모리: 설정 x 데이터셋 x 장수, 부류 한 번 평균 (Q2 실행분만)
q2 = pd.concat([pd.read_csv(p) for p in q2_paths], ignore_index=True)
display(q2.groupby(["dataset", "setting", "n_train_requested"])[
    ["n_train", "bank_size", "fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]].mean().round(1))
print("더한 시간(시간):", round((q2.fit_sec + q2.predict_sec + q2.metric_sec).sum() / 3600, 1))

## 7. 결과를 레포로

Drive `beyond_mvtec_results/`의 `q2_mvtec_ad.csv`, `q2_real_iad.csv`, `q2_visa.csv`와 실행한 이 노트북을 바탕화면에 내려받는다. `q2_scores/`는 올리지 않는다.